In [1]:
from pyspark.sql.functions import row_number
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    lag, col, lit, radians, sin, cos, asin, sqrt
)
from pyspark.sql.column import Column
from pyspark.sql.window import Window
from haversine import haversine, Unit
from dataclasses import dataclass
from pyspark import SparkContext
from datetime import datetime
import numpy as np
import pandas as pd
import os

In [2]:
@dataclass
class Coordinates:
    latitude: np.float64
    longitude: np.float64
    
    @property
    def point(self) -> tuple:
        return (
            self.latitude,
            self.longitude
        )

In [3]:
SOG_MOVE = 0.5
# 50-nautical-mile
NAUTICAL_MILE = 50
COLLISSION_METERS = 10


<h3>Big data analytics Task 4</h3>

In [4]:
data_source_folder = "aisdk-2021-12"
paths = [
    os.path.join(
        data_source_folder, 
        data_source_folder + "-" + str(i).rjust(2, "0") + ".csv"
    )
    for i in range(1, 32)
]

print("Using file names")
print(paths[:3])
print("....")

Using file names
['aisdk-2021-12/aisdk-2021-12-01.csv', 'aisdk-2021-12/aisdk-2021-12-02.csv', 'aisdk-2021-12/aisdk-2021-12-03.csv']
....


In [5]:
spark = SparkSession.builder.appName("task_4_cluster").getOrCreate()
if spark:
    print(f"Task_4 cluster working. Version = {spark.version}")
else:
    print("ERROR: something went wrong")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/05 01:12:10 WARN Utils: Your hostname, homedev-25p, resolves to a loopback address: 127.0.1.1; using 192.168.0.104 instead (on interface wlp0s20f3)
26/06/05 01:12:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/05 01:12:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Task_4 cluster working. Version = 4.1.2


In [6]:
# spark.stop()

start_time = datetime.now()
start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
print(f"Reading data to Apache Spark start time: {start_time_str}")

data = spark.read.csv(
    paths,
    sep=",",
    inferSchema=True,
    header=True
)

end_time = datetime.now()
end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")

time_diff_seconds = (end_time - start_time).seconds

print("----------------------")
print(f"Total {time_diff_seconds} seconds")
print(f"Execution end-time: {end_time_str}")


Reading data to Apache Spark start time: 2026-06-05 01:12:13


----------------------
Total 226 seconds
Execution end-time: 2026-06-05 01:15:59


<h3>Pre-processing</h3>

<h3>Data cleaning</h3>

In [7]:
data_clean = data.na.drop()
data_clean

DataFrame[# Timestamp: string, Type of mobile: string, MMSI: int, Latitude: double, Longitude: double, Navigational status: string, ROT: double, SOG: double, COG: double, Heading: int, IMO: string, Callsign: string, Name: string, Ship type: string, Cargo type: string, Width: int, Length: int, Type of position fixing device: string, Draught: double, Destination: string, ETA: string, Data source type: string, A: int, B: int, C: int, D: int]

In [8]:
data_clean = data_clean.withColumnRenamed("# Timestamp", "timestamp")

In [9]:
w = Window.partitionBy("MMSI").orderBy("timestamp")

In [10]:
data_clean = data_clean.withColumn("rn", row_number().over(w))

In [11]:
# data_clean.show(10, truncate=False)

In [12]:
# filter out stationary vessels

In [13]:
# filter out non stationary vessels
df_filtered = data_clean.filter(
    col("SOG") > SOG_MOVE
)

In [14]:
COORDINATE_CENTER = Coordinates(
    latitude=55.225000,
    longitude=14.245000
)
COORDINATE_CENTER

Coordinates(latitude=55.225, longitude=14.245)

In [15]:
def haversine_nm_value(
    a_lat_col: str, 
    a_lon_col: str, 
    b_lat_value: np.float64, 
    b_lon_value: np.float64
):
    return (
        3440.065 * 2 * asin(
            sqrt(
                sin((radians(a_lat_col) - radians(lit(b_lat_value))) / 2) ** 2 +
                cos(radians(lit(b_lat_value))) *
                cos(radians(a_lat_col)) *
                sin((radians(a_lon_col) - radians(lit(b_lon_value))) / 2) ** 2
            )
        )
    )
    
def haversine_meters(
    a_lat_col: Column,
    a_lon_col: Column,
    b_lat_col: Column,
    b_lon_col: Column
) -> Column:
    return (
        6371000 * 2 * asin(
            sqrt(
                sin((radians(a_lat_col) - radians(b_lat_col)) / 2) ** 2 +
                cos(radians(b_lat_col)) *
                cos(radians(a_lat_col)) *
                sin((radians(a_lon_col) - radians(b_lon_col)) / 2) ** 2
            )
        )
    )

In [16]:
# filter out 50 neutilon miles
# df_filtered = data_clean.filter(
#     haversine(
#         (col("Latitude"), col("Longitude")),               
#         COORDINATE_CENTER.point, 
#         unit=Unit.NAUTICAL_MILES
#     ) < NAUTICAL_MILE
# )
df_filtered = data_clean.filter(
    haversine_nm_value(
        "Latitude", "Longitude",
        COORDINATE_CENTER.latitude, 
        COORDINATE_CENTER.longitude
    ) < NAUTICAL_MILE
)

<h3>Simple data exploration</h3>

<h3>Collision analysis</h3>

In [17]:
collided_vessels = (
    df_filtered.alias("a")
      .join(
          df_filtered.alias("b"),
          (col("a.timestamp") == col("b.timestamp")) &
          (col("a.MMSI") < col("b.MMSI")) &
          (
            haversine_meters(
                col("a.Latitude"), col("a.Longitude"),
                col("b.Latitude"), col("b.Longitude")
            ) < COLLISSION_METERS
          )
      )
)

In [18]:
# collided_vessels.show(10, truncate=False)

In [19]:
print(collided_vessels.rdd.getNumPartitions())
print(spark.sparkContext.defaultParallelism)

26/06/05 01:16:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[509.321s][warning][gc,alloc] Executor task launch worker for task 3.0 in stage 4.0 (TID 860): Retried waiting for GCLocker too often allocating 131074 words


26/06/05 01:20:39 ERROR Executor: Exception in task 3.0 in stage 4.0 (TID 860)
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.util.collection.unsafe.sort.UnsafeSorterSpillReader.<init>(UnsafeSorterSpillReader.java:54)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeSorterSpillWriter.getReader(UnsafeSorterSpillWriter.java:159)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.getSortedIterator(UnsafeExternalSorter.java:581)
	at org.apache.spark.sql.execution.UnsafeExternalRowSorter.sort(UnsafeExternalRowSorter.java:174)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage3.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at org.apache.spark.sql.execution.window.WindowEv

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

In [20]:
spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
# The objective of this examination is to 
# evaluate your ability to process large-scale temporal and spatial data. 
# You are required to identify two vessels that have collided 
# (or experienced the closest possible physical proximity indicating a collision) 
# within a specified marine area. You must visualize their respective trajectories 
# 10 minutes prior to and 10 minutes following the time of collision.

In [ ]:
# Geographic Area: 
# 1. Filter the dataset to isolate vessels operating 
# within a 50-nautical-mile (nm) radius of a center coordinate 
# located at Latitude: 55.225000, Longitude: 14.245000.

# 2. Vessel State: You are looking specifically for moving vessels 
# that intersect in time and space, resulting in a collision.
# You must implement logic to identify and filter out stationary 
# vessels (e.g., ships at anchor or safely docked adjacent to one another).

# 3. Data Integrity: AIS data frequently contains errors. 
# You must account for and filter out GPS anomalies and data noise. 
# This is critical to ensure that a sudden jump in GPS coordinates 
# is not falsely identified as a collision.
